# Project 5 - Ensemble Machine Learning Using the Wine Quality Dataset
**Author:** Kellie Leopold  
**Date:** November 21, 2025  
**Objective:** Predict a Categorical Target and Evaluate Performance on the Titanic Dataset


## Introduction
In this lab we explore ensemble machine learning models using the Wine Quality dataset to see how combining multiple models can improve prediction accuracy. Ensemble methods such as Random Forests, Boosted Trees, and Voting Classifiers help reduce overfitting and often perform better than individual models by using their combined strengths. We use cross validation to obtain more reliable performance estimates and evaluate each model with accuracy, precision, recall, and F1 score. By comparing train and test performance and looking for small gaps between them, we can identify which ensemble models generalize well and provide the most accurate predictions of wine quality based on physicochemical properties.

## Imports

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
    VotingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

## Section 1. Load and Explore the Data
Load the titanic dataset directly from the seaborn library.

In [23]:
# Load the dataset (download from UCI and save in the same folder)
df = pd.read_csv("winequality-red.csv", sep=";")

# Display structure and first few rows
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Section 2. Data Exploration and Preparation
Transform the quality scores (0-10) into three categories to make classification more feasible:

Low quality: scores 3-4
Medium quality: scores 5-6
High quality: scores 7-8

In [24]:
# Create the quality_label column
def quality_to_label(q):
    if q <= 4:
        return "low"
    elif q <= 6:
        return "medium"
    else:
        return "high"


# Call the apply() method on the quality column to create the new quality_label column
df["quality_label"] = df["quality"].apply(quality_to_label)


# Create a numeric column for modeling: 0 = low, 1 = medium, 2 = high
def quality_to_number(q):
    if q <= 4:
        return 0
    elif q <= 6:
        return 1
    else:
        return 2
    
df["quality_numeric"] = df["quality"].apply(quality_to_number)

**Section 2 Reflection:**
In this step, we simplify the original wine quality scores into three categories—low, medium, and high—by creating the quality_label column. This makes the classification task more manageable and easier to interpret. We also create a numeric version, quality_numeric, so the data can be used directly with machine learning models that require numeric targets. Overall, these transformations prepare the dataset for both analysis and modeling while keeping the categories meaningful.

## Section 3. Feature Selection and Justification
Define multiple combinations of features to use as inputs to predict fare.

In [25]:
# Define input features (X) and target (y)
# Features: all columns except 'quality' and 'quality_label' and 'quality_numeric' - drop these from the input array
# Target: quality_label (the new column we just created)
# Check the column names
print(df.columns)

# Define input features (X) and target (y)
X = df.drop(columns=["quality", "quality_label", "quality_numeric"])
y = df["quality_numeric"]

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'quality_label',
       'quality_numeric'],
      dtype='object')


**Section 3 Reflection:**
We separate the dataset into input features (X) and target (y) to prepare for modeling. The features include all physicochemical properties of the wine, while the target is quality_numeric, the simplified numeric version of wine quality. This setup allows the model to learn how the measurable characteristics influence quality without using the answer as input.

## Section 4. Split the Data into Train and Test

In [26]:
# Train/test split (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Section 5. Evaluate Model Performance (Choose 2)
Take the best-performing case and explore other regression models.

In [27]:
# Initialize a list to store results
results = []

# Define models to evaluate
models = [
    ("AdaBoost (100)", AdaBoostClassifier(n_estimators=100, random_state=42)),
    ("MLP Classifier", MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42))
]

# Evaluate each model
for name, model in models:
    # Fit model
    model.fit(X_train, y_train)
    
    # Predict on train and test sets
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Get classification reports
    train_report = classification_report(y_train, y_train_pred, output_dict=True, zero_division=0)
    test_report = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    
    # Append results with train/test metrics and gaps
    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, y_train_pred),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Train F1": train_report["weighted avg"]["f1-score"],
        "Test F1": test_report["weighted avg"]["f1-score"],
        "Accuracy Gap": accuracy_score(y_train, y_train_pred) - accuracy_score(y_test, y_test_pred),
        "F1 Gap": train_report["weighted avg"]["f1-score"] - test_report["weighted avg"]["f1-score"]
    })

## Section 6. Compare Results

In [28]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Sort by Test Accuracy descending
results_df = results_df.sort_values(by="Test Accuracy", ascending=False)

# Display the table
print("\nSummary of All Models (sorted by Accuracy):")
display(results_df)


Summary of All Models (sorted by Accuracy):


,Model,Train Accuracy,Test Accuracy,Train F1,Test F1,Accuracy Gap,F1 Gap
1,MLP Classifier,0.848319,0.84375,0.818023,0.811507,0.004569,0.006516
0,AdaBoost (100),0.834246,0.82500,0.820863,0.815803,0.009246,0.005060


## Section 7. Conclusions and Insights
Predicting red wine quality is important for winemakers and quality control, as it helps identify wines likely to meet consumer expectations. In this analysis, the MLP Classifier achieved a test accuracy of 84.4% with minimal train-test gaps (Train Accuracy 84.8%, Train F1 81.8%), indicating strong generalization. AdaBoost (100 estimators) achieved a test accuracy of 82.5%, also with very small gaps, demonstrating reliable performance.

Comparing these results with other ensemble methods, Random Forest (100) achieved a higher test accuracy of 88.8%, while Random Forest (200, max_depth=10) reached 88.1%, and Gradient Boosting (100) achieved 85.6%. Although Random Forest attained slightly higher accuracy, it also exhibited larger train-test gaps (Train Accuracy 100%, Train F1 86.6%), suggesting some overfitting. Gradient Boosting performed comparably to the MLP Classifier with moderate generalization gaps. These observations highlight that averaging multiple decision trees can improve predictive accuracy but may increase overfitting risk, whereas boosting methods and multilayer classifiers provide more balanced generalization.

Future improvements could include feature engineering, addressing class imbalances, or exploring additional ensemble combinations to further enhance predictive performance. Overall, both boosting and bagging approaches, along with multilayer classification, provide robust predictions and actionable insights for understanding red wine quality.

Credit for insights in the conclusion: 
* [Adrianna Webb – Project 5 Notebook](https://github.com/AdriannaWebb/applied-ml-webb/blob/main/notebooks/project05/ml05_webb.ipynb)
* [Deb St. Cyr – Project 5 Notebook](https://github.com/14dstcyr/applied-ml-dstcyr/blob/main/notebooks/project05/ensemble_stcyr.ipynb)
